# PixiJoy neural inpaint — Colab GPU

Full UNet train → TFLite export → contract verify. **Runtime → Change runtime type → GPU** then **Runtime → Run all**.

When finished, download `dist/inpaint.tflite` and attach to [Release- ml-packs-v1](https://github.com/Saikumar-S0906/Release-/releases/tag/ml-packs-v1).

In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))

In [ ]:
import os, urllib.request, zipfile
from pathlib import Path

ROOT = Path("/content/pixijoy_inpaint")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# Prefer Release- zip; fall back to writing files if fetch fails.
ZIP_URL = "https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/inpaint_lab.zip"
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, "inpaint_lab.zip")
    with zipfile.ZipFile("inpaint_lab.zip") as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
    print("extracted lab from Release- main", ok)
except Exception as e:
    print("zip fetch failed:", e)
assert ok, "Could not load inpaint_lab.zip from Release- main"


In [ ]:
# Full budget — GPU auto batch=16 via config.resolved_batch()
!python -u train.py
!python -u export_tflite.py
!python -u verify_contract.py
from pathlib import Path
import hashlib
p = Path("dist/inpaint.tflite")
print("READY", p, p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())

In [ ]:
from google.colab import files
files.download("dist/inpaint.tflite")
print("Upload this file to GitHub Release ml-packs-v1 on Saikumar-S0906/Release-")